# 📉 **Evaluación Sumativa Unidad 02: Inferencia Estadística y Diagnóstico Paramétrico (ABP)**
**Asignatura:** Teoría de la Distribución y Probabilidad  
**Estudiante:** Matias Sebastian Labanda Pineda  
**Carrera:** Computación (2do Ciclo "A")  
**Dataset Regional:** `datos_loja.csv` (16 Cantones de la Provincia de Loja)  
**Variable de Estudio:** `Sin_Alcantarillado` (Habitantes sin acceso a la red pública de saneamiento)

---

## 📁 **0. Inicialización del Entorno y Carga de Datos**
A continuación, se importan las librerías científicas requeridas y se realiza la carga y segmentación del conjunto de datos. Para cumplir con la comparación de múltiples grupos, los 16 cantones se clasifican de forma académica en tres zonas geográficas según su ubicación: **Norte**, **Centro** y **Sur**.

In [3]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Carga directa del archivo subido a Google Colab
df = pd.read_csv('datos_loja.csv')

# Mapeo académico en subgrupos geográficos basado en tus datos reales
zonas_mapping = {
    'Loja': 'Centro', 'Catamayo': 'Centro', 'Chaguarpamba': 'Norte', 'Saraguro': 'Norte',
    'Paltas': 'Norte', 'Alamor': 'Norte', 'Olmedo': 'Norte', 'Pindal': 'Norte', 'Puyango': 'Norte',
    'Cariamanga': 'Sur', 'Macara': 'Sur', 'Celica': 'Sur', 'Zapotillo': 'Sur',
    'Espindola': 'Sur', 'Gonzanama': 'Sur', 'Quilanga': 'Sur'
}
df['Zona'] = df['Canton'].map(zonas_mapping)

print("--- DATASET REGIONAL DE LA PROVINCIA DE LOJA CARGADO DESDE CSV ---")
print(df.to_string(index=False))

--- DATASET REGIONAL DE LA PROVINCIA DE LOJA CARGADO DESDE CSV ---
      Canton  Poblacion  Viviendas  Sin_Alcantarillado   Zona
        Loja     250028      85412                9822 Centro
    Catamayo      35240      12150                3850 Centro
  Cariamanga      29111       9800                4100    Sur
      Macara      26042       8900                3200    Sur
     Puyango      22841       7500                2900  Norte
    Saraguro      18215       6200                2100  Norte
      Celica      16257       5100                1900    Sur
      Paltas      14571       4800                1800  Norte
   Zapotillo      14379       4600                1700    Sur
Chaguarpamba      14119       4500                1650  Norte
   Espindola      12247       3900                1500    Sur
   Gonzanama      10409       3400                1200    Sur
      Alamor       6970       2200                 800  Norte
      Olmedo       6857       2100                 750  Norte
   

## 🔬 **1. Pruebas de Hipótesis Unimuestrales (Frontera APE 09)**

Definimos una prueba de hipótesis para evaluar si la media poblacional ($\mu$) de habitantes sin alcantarillado por cantón difiere de un valor crítico referencial de $H_0: \mu_0 = 2000$ habitantes. Debido a las restricciones de nuestros datos reales (tamaño de muestra pequeño $n = 16$ y varianza poblacional $\sigma^2$ desconocida), el test paramétrico matemáticamente correcto es la **Distribución T de Student**.

### **Formulación de Hipótesis:**
* **Hipótesis Nula ($H_0$):** La media poblacional de habitantes sin alcantarillado en los cantones de Loja es igual a 2000.
$$H_0: \mu = 2000$$
* **Hipótesis Alterna ($H_1$):** La media poblacional de habitantes sin alcantarillado en los cantones de Loja es diferente de 2000 (Prueba de dos colas).
$$H_1: \mu \neq 2000$$

### **Criterio de Decisión:**
Se rechazará $H_0$ si el valor-$p < \alpha$, con un nivel de significancia establecido de $\alpha = 0.05$.

In [2]:
# Definición del parámetro hipotético nulo
mu_0 = 2000

# Ejecución de la prueba paramétrica T de Student unimuestral mediante scipy.stats
t_stat, p_val_unimuestral = stats.ttest_1samp(df['Sin_Alcantarillado'], popmean=mu_0)

# Parámetros descriptivos auxiliares para la justificación
n = len(df)
grados_libertad = n - 1
media_muestral = df['Sin_Alcantarillado'].mean()
error_estandar = stats.sem(df['Sin_Alcantarillado'])

print("--- RESULTADOS DE LA PRUEBA T UNIMUESTRAL ---")
print(f"Media Muestral Observada (x_bar): {media_muestral:.2f}")
print(f"Error Estándar de la Media (SE): {error_estandar:.4f}")
print(f"Grados de Libertad (df = n - 1): {grados_libertad}")
print(f"Métrica del Estadístico t: {t_stat:.4f}")
print(f"Valor-p obtenido computacionalmente: {p_val_unimuestral:.6f}")

# Justificación estadística automatizada
alpha = 0.05
print("\n[JUSTIFICACIÓN TÉCNICA]:")
if p_val_unimuestral < alpha:
    print(f"Como el valor-p ({p_val_unimuestral:.6f}) < alpha ({alpha}), existe evidencia estadística suficiente")
    print(f"para RECHAZAR la hipótesis nula (H₀). La media real difiere significativamente de {mu_0}.")
else:
    print(f"Como el valor-p ({p_val_unimuestral:.6f}) >= alpha ({alpha}), NO se rechaza la hipótesis nula (H₀).")
    print(f"Las variaciones observadas pueden atribuirse puramente al azar muestral.")

--- RESULTADOS DE LA PRUEBA T UNIMUESTRAL ---
Media Muestral Observada (x_bar): 2382.62
Error Estándar de la Media (SE): 570.2750
Grados de Libertad (df = n - 1): 15
Métrica del Estadístico t: 0.6709
Valor-p obtenido computacionalmente: 0.512453

[JUSTIFICACIÓN TÉCNICA]:
Como el valor-p (0.512453) >= alpha (0.05), NO se rechaza la hipótesis nula (H₀).
Las variaciones observadas pueden atribuirse puramente al azar muestral.


## 📊 **2. Comparación de Grupos mediante ANOVA de 1 Factor (Frontera APE 10)**

Para verificar si existen brechas significativas en el acceso al alcantarillado según el sector geográfico de la provincia, dividimos el dataset en 3 subgrupos: **Norte** ($n_1=7$), **Centro** ($n_2=2$) y **Sur** ($n_3=7$).

Para comparar múltiples medias simultáneamente sin inflar la probabilidad de cometer un Error Tipo I (falso positivo), implementamos un **Análisis de Varianza (ANOVA de 1 factor)** complementado con una prueba **Post-Hoc de Tukey**.

### **Formulación de Hipótesis para el ANOVA:**
* **Hipótesis Nula ($H_0$):** Los promedios poblacionales de habitantes sin alcantarillado son estadísticamente homogéneos e iguales en las tres zonas de Loja.
$$H_0: \mu_{\text{Norte}} = \mu_{\text{Centro}} = \mu_{\text{Sur}}$$
* **Hipótesis Alterna ($H_1$):** Al menos una zona geográfica presenta un promedio de habitantes sin alcantarillado significativamente diferente a las demás.
$$H_1: \exists \quad \mu_i \neq \mu_j \quad \text{para algún } i \neq j$$

In [6]:
# 1. Extracción y segmentación de los subgrupos de análisis
grupo_norte = df[df['Zona'] == 'Norte']['Sin_Alcantarillado']
grupo_centro = df[df['Zona'] == 'Centro']['Sin_Alcantarillado']
grupo_sur = df[df['Zona'] == 'Sur']['Sin_Alcantarillado']

print("--- VALIDACIÓN PREVIA DE SUPUESTOS PARAMÉTRICOS ---")

# SOLUCIÓN CIENTÍFICA AL NaN: Evaluación de la normalidad a través de los residuos globales del modelo
residuos = np.concatenate([
    grupo_norte - grupo_norte.mean(),
    grupo_centro - grupo_centro.mean(),
    grupo_sur - grupo_sur.mean()
])

stat_shapiro, p_val_shapiro = stats.shapiro(residuos)
status_normalidad = "Se cumple supuesto (p > 0.05)" if p_val_shapiro > 0.05 else "No se cumple"
print(f"[Shapiro-Wilk Global] Normalidad de Residuos | Estadístico: {stat_shapiro:.4f} | valor-p: {p_val_shapiro:.4f} | {status_normalidad}")

# Supuesto 2: Homocedasticidad mediante Levene usando la mediana (robusto ante asimetría)
stat_lev, p_val_lev = stats.levene(grupo_norte, grupo_centro, grupo_sur, center='median')
status_homocedasticidad = "Varianzas Homogéneas (p > 0.05)" if p_val_lev > 0.05 else "Heterocedasticidad detectada (p <= 0.05)"
print(f"[Levene] Verificación de varianzas: p-value = {p_val_lev:.4f} | {status_homocedasticidad}\n")

print("--- EJECUCIÓN DEL ALGORITMO ANOVA DE 1 FACTOR ---")
# Parámetros del diseño experimental computacional
k = df['Zona'].nunique()   # Número de grupos (3)
n_tot = len(df)            # Muestra total (16)
df_entre = k - 1           # Grados de libertad del numerador
df_dentro = n_tot - k      # Grados de libertad del denominador

# Uso de la abstracción avanzada de scipy.stats
f_stat, p_val_anova = stats.f_oneway(grupo_norte, grupo_centro, grupo_sur)

print(f" -> Grados de Libertad Entre Grupos (df_entre): {df_entre}")
print(f" -> Grados de Libertad Dentro de Grupos (df_dentro): {df_dentro}")
print(f" -> Métrica del Estadístico F: {f_stat:.4f}")
print(f" -> VALOR-P ARROJADO POR ANOVA: {p_val_anova:.6f}")

if p_val_anova < 0.05:
    print("\n CONCLUSIÓN TÉCNICA: valor-p < 0.05 -> Se RECHAZA la Hipótesis Nula (H₀).")
    print("                      Existen diferencias significativas. Es obligatorio aplicar la prueba Post-Hoc de Tukey.")
else:
    print("\n CONCLUSIÓN TÉCNICA: valor-p >= 0.05 -> No se rechaza H₀. Las medias son estadísticamente equivalentes.")

--- VALIDACIÓN PREVIA DE SUPUESTOS PARAMÉTRICOS ---
[Shapiro-Wilk Global] Normalidad de Residuos | Estadístico: 0.9790 | valor-p: 0.9551 | Se cumple supuesto (p > 0.05)
[Levene] Verificación de varianzas: p-value = 0.0034 | Heterocedasticidad detectada (p <= 0.05)

--- EJECUCIÓN DEL ALGORITMO ANOVA DE 1 FACTOR ---
 -> Grados de Libertad Entre Grupos (df_entre): 2
 -> Grados de Libertad Dentro de Grupos (df_dentro): 13
 -> Métrica del Estadístico F: 9.4441
 -> VALOR-P ARROJADO POR ANOVA: 0.002931

 CONCLUSIÓN TÉCNICA: valor-p < 0.05 -> Se RECHAZA la Hipótesis Nula (H₀).
                      Existen diferencias significativas. Es obligatorio aplicar la prueba Post-Hoc de Tukey.


### **3. Análisis de Comparaciones Múltiples en Pares (Post-Hoc Tukey)**

Dado que el ANOVA arrojó un valor-$p$ de significancia estadística, se utiliza la abstracción de `statsmodels` para ejecutar la prueba de Tukey. Esto nos permitirá identificar de forma rigurosa qué pares específicos de zonas geográficas provocan el rechazo de la igualdad de medias.

In [5]:
print("--- PRUEBA POST-HOC DE TUKEY (PAIRS COMPARISON) ---")
# Implementación avanzada con statsmodels
tukey_result = pairwise_tukeyhsd(endog=df['Sin_Alcantarillado'],
                                 groups=df['Zona'],
                                 alpha=0.05)
print(tukey_result)

--- PRUEBA POST-HOC DE TUKEY (PAIRS COMPARISON) ---
    Multiple Comparison of Means - Tukey HSD, FWER=0.05     
group1 group2  meandiff  p-adj    lower      upper    reject
------------------------------------------------------------
Centro  Norte -5343.1429 0.0025   -8655.28 -2031.0058   True
Centro    Sur    -4836.0 0.0053 -8148.1371 -1523.8629   True
 Norte    Sur   507.1429 0.8191 -1700.9485  2715.2343  False
------------------------------------------------------------
